# Demo: BERTScore vs BLEU and ROUGE

**Week 6, Module 02, Segment 5 (slides 36 to 38)** | Cordwell Home and Hardware | ~20 to 25 minutes

Cordwell's support team runs a model that summarizes customer tickets. In Segments 2 through 4 we scored its outputs with BLEU and ROUGE and watched both metrics fail on paraphrase: they compare token surfaces, so a synonym and an antonym scored identically. This demo picks up exactly where slide 31 left off and asks whether BERTScore, which compares contextual embeddings instead of tokens, does better. It does, on one specific axis. It also fails on another, and knowing which is which is the point of the session.

**What you will see**

1. Recap: the minimal pair that broke BLEU and ROUGE (executed, verified numbers)
2. BERTScore on the same pair: embeddings separate what n-grams cannot
3. Why `rescale_with_baseline=True` is not optional
4. The Cordwell Disagreement revisited: what BERTScore fixes and what it does not
5. Optional: the greedy alignment heatmap
6. Side-by-side: when to reach for which metric

**Environment**: sacrebleu 2.6.0, rouge-score 0.1.2, bert-score 0.3.13, torch 2.13.0. No API keys, no LLM backend. The BERTScore cells download a HuggingFace model on first run (see the model note in Part 2), so run this notebook once before class to warm the cache.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# MPS fallback must be set before torch is imported
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import importlib.metadata as md

import torch
from sacrebleu.metrics import BLEU, CHRF
from rouge_score import rouge_scorer


def pick_device() -> str:
    """Runtime device selection: CUDA, then Apple MPS, then CPU."""
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device()

for pkg in ("sacrebleu", "rouge-score", "bert-score", "torch"):
    print(f"{pkg:12s} {md.version(pkg)}")
print(f"device       {DEVICE}")

## Part 1: Recap, the pair that broke the n-gram metrics

Slide 31's minimal pair. One candidate swaps in a synonym, the other an antonym. Same reference, one word apart, opposite meanings.

> **Ask the room before running**: which candidate should a competent metric score higher, and by roughly how much?

In [ ]:
# A one-word swap on a state, not a size: same-axis size antonyms (large/small)
# sit too close in embedding space for BERTScore to separate, even with roberta-large.
# reference = "Use the large drill bit for masonry."
# cand_syn  = "Use the big drill bit for masonry."    # synonym: meaning preserved
# cand_ant  = "Use the small drill bit for masonry."  # antonym: meaning inverted

reference = "The safety guard is installed."
cand_syn  = "The safety guard is mounted."   # synonym: meaning preserved
cand_ant  = "The safety guard is missing."   # antonym: meaning inverted

bleu   = BLEU()
chrfpp = CHRF(word_order=2)  # chrF++

# NOTE the argument-order trap from slide 23:
# rouge_scorer.score() takes TARGET (reference) FIRST, prediction second.
# sacrebleu takes candidates first. Reversed mental orders, silent errors.
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

print(f"{'candidate':10s} {'BLEU':>7s} {'chrF++':>7s} {'R-1 F':>7s} {'R-2 F':>7s}")
for name, cand in [("synonym", cand_syn), ("antonym", cand_ant)]:
    b = bleu.corpus_score([cand], [[reference]]).score
    c = chrfpp.corpus_score([cand], [[reference]]).score
    r = rouge.score(reference, cand)  # target first
    print(f"{name:10s} {b:7.2f} {c:7.2f} "
          f"{r['rouge1'].fmeasure * 100:7.2f} {r['rouge2'].fmeasure * 100:7.2f}")

print()
print(bleu.get_signature())

**Read the table.** BLEU: 53.73 for both. ROUGE-1: 80.00 for both. chrF++ separates them by 0.5 points, which is noise. Meaning preserved and meaning inverted are indistinguishable, because every one of these metrics tokenizes and counts overlap. `installed`, `mounted`, and `missing` are simply three different tokens.

This is not a weakness at the edges. These metrics are structurally unable to represent meaning. That is the gap BERTScore claims to close.

## Part 2: BERTScore on the same pair

From slide 36: for each candidate token, BERTScore finds the closest reference token by cosine similarity over **contextual embeddings**, then aggregates to precision, recall, and F1. `mounted` and `installed` live in nearly the same embedding neighborhood; `missing` does not. That is the entire bet.

**Model note.** `roberta-large` is the library default for English and what the slides show. It is a ~1.4 GB download on first use and takes noticeably longer to load than to score on the cohort's no-GPU Macs. `distilbert-base-uncased` (~260 MB) is the sanctioned lighter fallback; both models ship rescale baselines, which Part 3 explains. The first cell below verifies that the baseline files exist locally so a typo'd model name fails here, loudly, instead of mid-demo.


In [ ]:
import bert_score
from bert_score import score
import os

from pathlib import Path

BERTSCORE_MODEL = str(Path("~/models/distilbert-base-uncased").expanduser().resolve())  # local model, ~420 MB

# The rescale baselines ship inside the bert-score *package*, not with the model.
# They are keyed by the canonical model name, so a local path must be reduced to
# its basename before it will match a bundled file.
BASELINE_NAME = os.path.basename(BERTSCORE_MODEL)  # e.g. "distilbert-base-uncased"
baseline = os.path.join(
    os.path.dirname(bert_score.__file__),
    "rescale_baseline", "en", f"{BASELINE_NAME}.tsv",
)
assert os.path.exists(baseline), f"No rescale baseline bundled for {BASELINE_NAME}"
print(f"rescale baseline found: .../{os.path.relpath(baseline, os.path.dirname(bert_score.__file__))}")

In [ ]:
# First run downloads the model from HuggingFace; later runs hit the local cache.
P, R, F1 = score(
    cands=[cand_syn, cand_ant],
    refs=[reference, reference],
    lang="en",                    # required for rescaling to resolve a baseline
    model_type=BERTSCORE_MODEL,
    num_layers=5,                 # distilbert-base-uncased: bert-score's validated layer (of 6); required for a local path
    rescale_with_baseline=True,   # Part 3 shows why this is not optional
    baseline_path=baseline,       # local path can't self-resolve the bundled baseline
    device=DEVICE,
)

for name, f in zip(["synonym", "antonym"], F1.tolist()):
    print(f"{name:10s} BERTScore F1 = {f:.4f}")

> **NOTE** The build environment blocks HuggingFace downloads, so this cell cannot be executed as-is on cohort machines. Access to the model (downloaded for offline access as an alternative).

**Contrast with Part 1 while it runs**: BLEU and ROUGE gave these two candidates *identical* scores. Whatever the exact values above, the fact that they differ at all is the headline.

## Part 3: Why `rescale_with_baseline=True` is not optional

Slide 38's trap. Raw BERTScore with RoBERTa-large compresses into a narrow band, roughly 0.85 to 0.95, regardless of quality. An unrelated sentence can score around 0.83, which reads like a B grade but is close to the floor. Rescaling applies a linear transform against a precomputed random-pairs baseline; the library authors document the WMT18 average dropping from about 0.93 to 0.58 under rescaling. After rescaling, 0.58 reads as what it is: below average.

The next cell scores the synonym, the antonym, and a sentence about Cordwell's earnings call that has nothing to do with the safety guard, both raw and rescaled.


In [ ]:
cand_unrelated = "The quarterly earnings call is scheduled for Tuesday."

cands = [cand_syn, cand_ant, cand_unrelated]
refs  = [reference] * 3
labels = ["synonym", "antonym", "unrelated"]

_, _, f1_raw = score(
    cands=cands, refs=refs, lang="en",
    model_type=BERTSCORE_MODEL, rescale_with_baseline=False, device=DEVICE,
    num_layers=5,
)
_, _, f1_rescaled = score(
    cands=cands, refs=refs, lang="en",
    model_type=BERTSCORE_MODEL, rescale_with_baseline=True, device=DEVICE,
    num_layers=5, baseline_path=baseline
)

print(f"{'candidate':11s} {'raw F1':>8s} {'rescaled F1':>12s}")
for lab, raw, resc in zip(labels, f1_raw.tolist(), f1_rescaled.tolist()):
    print(f"{lab:11s} {raw:8.4f} {resc:12.4f}")

**The talking point**: without rescaling you will misread garbage as a B grade. Always set `rescale_with_baseline=True`, and note it requires `lang` to be set so the library can resolve the right baseline file. This is a config-pinning lesson wearing a new metric's clothes: same theme as the ROUGE stemmer flag.

## Part 4: The Cordwell Disagreement revisited

Slide 30's break. One ticket, one reference, two system outputs. System A copies the reference's wording but inverts the resolution: refund instead of replacement. System B paraphrases heavily and gets the facts right. First, the n-gram verdict, re-executed live:

In [ ]:
cw_ref = ("Customer received a cracked 2200 PSI pressure washer "
          "and wants a replacement unit shipped before Saturday.")
cw_sysA = ("Customer received a cracked 2200 PSI pressure washer "
           "and wants a refund issued before Saturday.")           # factually WRONG
cw_sysB = ("The buyer's 2200 PSI power washer arrived damaged; "
           "they are asking for a replacement to arrive by Saturday.")  # correct

print(f"{'system':22s} {'BLEU':>7s} {'chrF++':>7s} {'R-1 F':>7s} {'R-2 F':>7s} {'R-L F':>7s}")
for name, cand in [("A (wrong resolution)", cw_sysA), ("B (correct)", cw_sysB)]:
    b = bleu.corpus_score([cand], [[cw_ref]]).score
    c = chrfpp.corpus_score([cand], [[cw_ref]]).score
    r = rouge.score(cw_ref, cand)
    print(f"{name:22s} {b:7.2f} {c:7.2f} "
          f"{r['rouge1'].fmeasure * 100:7.2f} "
          f"{r['rouge2'].fmeasure * 100:7.2f} "
          f"{r['rougeL'].fmeasure * 100:7.2f}")

Every n-gram metric prefers the factually wrong output by a landslide: BLEU 69.97 vs 6.89, ROUGE-1 83.87 vs 34.29. System B is being punished purely for paraphrasing. Now the same pair under BERTScore:

In [ ]:
P, R, F1 = score(
    cands=[cw_sysA, cw_sysB],
    refs=[cw_ref, cw_ref],
    lang="en",
    model_type=BERTSCORE_MODEL,
    rescale_with_baseline=True,
    baseline_path=baseline,
    num_layers=5,
    device=DEVICE,
)

for name, f in zip(["A (wrong resolution)", "B (correct)"], F1.tolist()):
    print(f"{name:22s} BERTScore F1 = {f:.4f}")


> - **What BERTScore fixes**: System B's score rises dramatically relative to its n-gram numbers. Paraphrase, morphological variation, and same-meaning word substitution now score as close. B is no longer punished for saying "power washer arrived damaged" instead of "cracked pressure washer."
> - **What BERTScore does not fix**: System A likely still scores well, quite possibly above B. "Refund" sits near "replacement" in embedding space the same way "mounted" sits near "installed". BERTScore compares against a reference by embedding proximity; it has no idea the resolution is inverted and no access to the source ticket.
>
> **BERTScore solved the paraphrase problem, not the factuality problem.** Something still has to read the source ticket, which is exactly where Segment 6 (LLM-as-a-judge) picks up.


## Part 5 (optional, if time allows): see the greedy alignment

`bert_score.plot_example` renders the pairwise cosine-similarity matrix behind the score: candidate tokens down one axis, reference tokens across the other, the greedy max highlighted. It makes the slide 36 formula concrete in one image. Worth two minutes if the room is tracking well; skip cleanly if not.

**Argument-order trap, again, in the same library family**: `plot_example` takes **candidate first, reference second**, which is the *reverse* of `rouge_scorer.score`. Say this out loud. Both orders exist in this ecosystem and neither call will warn you.

In [ ]:
from bert_score import plot_example

# candidate FIRST, reference second (reverse of rouge_scorer.score)
plot_example(
    cand_syn,
    reference,
    lang="en",
    model_type=BERTSCORE_MODEL,
    rescale_with_baseline=True,
    num_layers=5,
    baseline_path=baseline,
)

## Part 6: side by side, when to reach for which

| | BLEU | ROUGE | BERTScore |
|---|---|---|---|
| **Compares** | Token n-grams, precision-oriented, plus brevity penalty | Token n-grams and LCS, recall-oriented | Contextual token embeddings, greedy alignment, P and R and F1 |
| **Sees paraphrase as** | A mismatch | A mismatch | A near-match |
| **Sees an antonym swap as** | Same as a synonym swap | Same as a synonym swap | Different, usually penalized |
| **Sees a factual inversion as** | Invisible if surface-close | Invisible if surface-close | Largely invisible: still reference-proximity |
| **Model required** | No | No | Yes (~1.4 GB default, network on first run) |
| **Speed and cost** | Milliseconds, CPU trivial | Milliseconds, CPU trivial | Seconds per batch, model load dominates |
| **Config to pin** | Signature: tokenizer, smoothing, version | Stemmer flag, variant (rougeL vs rougeLsum), version | Model, layer, rescaling on, library version |
| **Granularity** | Corpus-level by construction | Per-pair | Per-pair |
| **Cordwell use** | CI tripwire on the blurb pipeline | Summarization regression vs own baseline | Paraphrase-heavy comparisons, reranking candidates |

Three closing points, each earned by a cell above:

1. **BERTScore is the first metric today that looked at meaning**, and it visibly separates the synonym from the antonym where BLEU and ROUGE were structurally blind.
2. **It is still a reference-proximity metric.** It rescued System B from paraphrase punishment but has no mechanism to catch System A's inverted resolution. Reference-based means reference-limited.
3. **Every metric so far has shipped with a config trap**: BLEU's smoothing, ROUGE's stemmer and variant, BERTScore's rescaling. The recurring engineering lesson is that an eval number without its pinned config is not a number.

**Hand-off question for Segment 6**: what kind of evaluator could catch System A? It would need to read the source ticket, not just the reference. That is where we go next.